<a href="https://colab.research.google.com/github/MuhammadAqsandy/scikit-learn-cookbook/blob/main/chapter_05_linear_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 5: Linear Models and Regularization
## 📌 Summary
Linear models adalah fondasi ML: sederhana, interpretable, dan efisien. Chapter ini mencakup **Linear Regression, Ridge (L2), Lasso (L1), ElasticNet**, dan **SGD**.

## 🧠 Theoretical Deep-Dive

### 5.1 Linear Regression
Minimize residual sum of squares: RSS = ||y - Xw||²
Solusi: w = (XᵀX)⁻¹Xᵀy (Normal Equation)

### 5.2 Regularization
Regularisasi menambahkan penalty pada koefisien besar untuk mencegah overfitting:

- **Ridge (L2)**: Loss = RSS + α·||w||²  
  → Shrinks koefisien tapi tidak nol. Semua fitur tetap
  
- **Lasso (L1)**: Loss = RSS + α·||w||₁  
  → Bisa set koefisien ke **nol** → otomatis feature selection
  
- **ElasticNet**: Loss = RSS + α·ρ·||w||₁ + α·(1-ρ)/2·||w||²  
  → Kombinasi L1+L2. `l1_ratio` mengontrol rasio

**Alpha (α):** semakin besar → regularisasi lebih kuat → model lebih simple

### 5.3 Stochastic Gradient Descent (SGD)
Update weight satu sample (atau mini-batch) per iterasi. Sangat efisien untuk **large-scale** datasets. Loss functions bisa diubah (hinge → SVM, log → Logistic Regression).

## 💻 Code Reproduction

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

X, y = make_regression(n_samples=200, n_features=5, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print("Intercept:", lr.intercept_.round(3))
print("Coefficients:", lr.coef_.round(3))
print("MSE:", round(mean_squared_error(y_test, y_pred), 2))
print("R²:", round(r2_score(y_test, y_pred), 4))

Intercept: 0.973
Coefficients: [ 2.834 11.015 64.885 17.945 69.867]
MSE: 85.06
R²: 0.9902


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

X, y = make_regression(n_samples=100, n_features=20, noise=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_s, y_train)
    r2 = ridge.score(X_test_s, y_test)
    print(f"alpha={alpha:6}: R²={r2:.4f}, max|coef|={np.abs(ridge.coef_).max():.3f}")

alpha= 0.001: R²=0.9647, max|coef|=93.210
alpha=  0.01: R²=0.9647, max|coef|=93.200
alpha=   0.1: R²=0.9647, max|coef|=93.110
alpha=   1.0: R²=0.9642, max|coef|=92.210
alpha=  10.0: R²=0.9484, max|coef|=84.060
alpha= 100.0: R²=0.6637, max|coef|=44.633


In [ ]:
from sklearn.linear_model import Lasso
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

X, y = make_regression(n_samples=100, n_features=20, n_informative=5, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

alphas = [0.001, 0.01, 0.1, 1.0, 10.0]
for alpha in alphas:
    lasso = Lasso(alpha=alpha, max_iter=5000)
    lasso.fit(X_train_s, y_train)
    n_zero = np.sum(lasso.coef_ == 0)
    r2 = lasso.score(X_test_s, y_test)
    print(f"alpha={alpha:5}: R²={r2:.4f}, zero_coefs={n_zero}/{X.shape[1]}")

alpha=0.001: R²=0.9690, zero_coefs=0/20
alpha= 0.01: R²=0.9692, zero_coefs=0/20
alpha=  0.1: R²=0.9708, zero_coefs=1/20
alpha=  1.0: R²=0.9814, zero_coefs=10/20
alpha= 10.0: R²=0.9293, zero_coefs=17/20


In [ ]:
from sklearn.linear_model import ElasticNet, RidgeCV, LassoCV
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X, y = make_regression(n_samples=200, n_features=15, n_informative=8, noise=15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# ElasticNet
en = ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=5000)
en.fit(X_train_s, y_train)
print("ElasticNet R²:", round(en.score(X_test_s, y_test), 4))

# Auto-tune with CV variants
ridge_cv = RidgeCV(alphas=[0.1, 1.0, 10.0])
ridge_cv.fit(X_train_s, y_train)
print("RidgeCV best alpha:", ridge_cv.alpha_)
print("RidgeCV R²:", round(ridge_cv.score(X_test_s, y_test), 4))

lasso_cv = LassoCV(cv=5, max_iter=5000, random_state=42)
lasso_cv.fit(X_train_s, y_train)
print("LassoCV best alpha:", round(lasso_cv.alpha_, 4))
print("LassoCV R²:", round(lasso_cv.score(X_test_s, y_test), 4))

ElasticNet R²: 0.9873
RidgeCV best alpha: 0.1
RidgeCV R²: 0.9912
LassoCV best alpha: 0.3425
LassoCV R²: 0.9909


In [9]:
from sklearn.linear_model import SGDRegressor, SGDClassifier
from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# SGD Regression
X, y = make_regression(n_samples=500, n_features=10, noise=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

sgd_reg = SGDRegressor(loss="squared_error", penalty="l2", alpha=0.01, max_iter=1000, random_state=42)
sgd_reg.fit(X_train_s, y_train)
print("SGD Regressor R²:", round(sgd_reg.score(X_test_s, y_test), 4))

# SGD Classification
Xc, yc = make_classification(n_samples=500, n_features=10, random_state=42)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, random_state=42)
scaler2 = StandardScaler()
Xc_train_s = scaler2.fit_transform(Xc_train)
Xc_test_s = scaler2.transform(Xc_test)

sgd_clf = SGDClassifier(loss="hinge", penalty="l2", alpha=0.0001, max_iter=1000, random_state=42)
sgd_clf.fit(Xc_train_s, yc_train)
print("SGD Classifier (SVM) accuracy:", round(sgd_clf.score(Xc_test_s, yc_test), 4))

SGD Regressor R²: 0.995
SGD Classifier (SVM) accuracy: 0.88


## ✅ Chapter Summary
- **Linear Regression**: baseline untuk regression tasks
- **Ridge (L2)**: best ketika semua fitur relevan, cegah multicollinearity
- **Lasso (L1)**: best untuk feature selection otomatis (sparse solution)
- **ElasticNet**: kombinasi keduanya, sering mengungguli keduanya
- **SGD**: untuk large-scale datasets yang tidak muat di memori
- Selalu gunakan **CV variants** (RidgeCV, LassoCV) untuk cari alpha optimal